# Bước 3: Data Analysis with SQL
## Khám Phá Dữ Liệu Qua Hệ Thống Truy Vấn SQLite 


## 3.0 Thiết Lập Môi Trường, Khung Chiến Lược & Ý Nghĩa Đối Với Dự Án

* **Data Pipeline tái sử dụng:** Dữ liệu sạch `customerData_clean.csv` (432.256 dòng) được lưu trữ bền vững tại `Data/customers.db`.



In [1]:
import sqlite3, pandas as pd
from pathlib import Path

p = Path('Data') if Path('Data').exists() else Path('../Data')
conn = sqlite3.connect(p / 'customers.db')
pd.read_csv(p / 'customerData_clean.csv').to_sql('customers', conn, index=False, if_exists='replace')

def run_sql(query):
    return pd.read_sql_query(query, conn)

---
# PHẦN 1:CÁC YẾU TỐ ĐI CÙNG VẤN ĐỀ GỐC

## Truy vấn 1: Tỷ lệ Chuyển đổi Chuẩn Toàn Sàn (Baseline Conversion Benchmark)

* **Vấn đề cốt lõi:** Trên toàn bộ 432.256 khách hàng hợp lệ, tỷ lệ chốt đơn mua hàng trung bình là bao nhiêu và tỷ lệ thất thoát (không mua) là bao nhiêu để làm mốc đối chuẩn (Baseline Benchmark)?

In [2]:
q1 = """
SELECT 
    COUNT(*) AS Tong_So_Khach,
    SUM(PurchaseStatus) AS So_Khach_Mua,
    COUNT(*) - SUM(PurchaseStatus) AS So_Khach_Khong_Mua,
    ROUND(AVG(PurchaseStatus) * 100.0, 2) AS Ty_Le_Chuyen_Doi_Chuan_Pct,
    ROUND((1.0 - AVG(PurchaseStatus)) * 100.0, 2) AS Ty_Le_That_Thoat_Dropoff_Pct
FROM customers;
"""
run_sql(q1)


,Tong_So_Khach,So_Khach_Mua,So_Khach_Khong_Mua,Ty_Le_Chuyen_Doi_Chuan_Pct,Ty_Le_That_Thoat_Dropoff_Pct
0,432256,187357,244899,43.34,56.66


**Kết luận :**
* **Kết quả định lượng:** Tỷ lệ chuyển đổi chuẩn của toàn sàn đạt **43.34%** (187.357 khách chốt đơn), trong khi nhóm **Không mua chiếm tới 56.66%** (244.899 khách).
* **Ý nghĩa chiến lược:** Con số **43.34%** là KPI nền tảng của sàn. Mọi phân tích sau đây đều lấy mốc 43.34% làm chuẩn mực đo lường.


---
## Truy vấn 2: Đánh Giá Hiệu Quả Kênh Tiếp Thị & Rủi Ro Lãng Phí Chi Phí CAC (ReferralSource ROI)

* **Câu hỏi nghiên cứu (RQ4 & CMO Persona):** Kênh tiếp cận nào mang lại tỷ lệ chuyển đổi cao nhất, và kênh nào đang gây lãng phí chi phí thu hút khách hàng (CAC)?

In [3]:
q2 = """
SELECT 
    ReferralSource AS Kenh_Tiep_Thi,
    COUNT(*) AS So_Luong_Khach,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) AS Ty_Trong_Khach_Pct,
    SUM(PurchaseStatus) AS So_Khach_Chot_Don,
    ROUND(AVG(PurchaseStatus) * 100.0, 2) AS Ty_Le_Chot_Don_CVR_Pct
FROM customers
GROUP BY ReferralSource
ORDER BY Ty_Le_Chot_Don_CVR_Pct DESC;
"""
run_sql(q2)


,Kenh_Tiep_Thi,So_Luong_Khach,Ty_Trong_Khach_Pct,So_Khach_Chot_Don,Ty_Le_Chot_Don_CVR_Pct
0,Referral,34368,7.95,15123,44.00
1,Organic,179859,41.61,78079,43.41
2,Social,56794,13.14,24599,43.31
3,Email,64542,14.93,27928,43.27
4,Paid Ads,96693,22.37,41628,43.05


> **Kết luận:**
> * **Phát hiện nghịch lý CAC:** Kênh giới thiệu tự nhiên (**Referral**) đạt CVR cao nhất sàn (**44.00%**), trong khi kênh quảng cáo trả phí (**Paid Ads**) chiếm tới 22.37% lượng khách (96.693 người) nhưng lại ghi nhận CVR thấp nhất sàn (**43.05%**).
> * **Quyết định cho Bước 6:** Cung cấp bằng chứng thực nghiệm để thiết lập quy tắc Kịch bản 3 trong Mô hình Học máy: Ngắt ngay lập tức ngân sách Paid Ads đối với nhóm khách hàng có xác suất mua dưới 30% để chống lãng phí chi phí tiếp thị.


---
# PHẦN 2:CÁC CÂU HỎI MỤC TIÊU KINH DOANH


## Truy vấn 3: Phân Khúc Khách Hàng & "Nghịch Lý Khách Hàng VIP" (CustomerSegment Analysis)

* **Câu hỏi nghiên cứu (RQ2 & Growth Lead / CEO):** Khách hàng phân khúc cao (VIP) có tỷ lệ mua sắm ở phiên hiện tại vượt trội hơn nhóm Regular và Premium hay không?
* **Ý nghĩa đối với Mô hình Học máy (Bước 6):** Cung cấp cơ sở định lượng giải thích vì sao biến `CustomerSegment_VIP` có hệ số âm trong Logistic Regression (Odds Ratio = 0.4619, $\\beta = -0.7725$ sau khi chuẩn hóa StandardScaler).

In [4]:
q3 = """
SELECT 
    CustomerSegment AS Phan_Khuc,
    COUNT(*) AS So_Luong_Khach,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) AS Ty_Trong_Khach_Pct,
    SUM(PurchaseStatus) AS So_Khach_Chot_Don,
    ROUND(AVG(PurchaseStatus) * 100.0, 2) AS Ty_Le_Chot_Don_CVR_Pct,
    ROUND(AVG(AnnualIncome), 2) AS Thu_Nhap_Binh_Quan_USD
FROM customers
GROUP BY CustomerSegment
ORDER BY Ty_Le_Chot_Don_CVR_Pct DESC;
"""
run_sql(q3)


,Phan_Khuc,So_Luong_Khach,Ty_Trong_Khach_Pct,So_Khach_Chot_Don,Ty_Le_Chot_Don_CVR_Pct,Thu_Nhap_Binh_Quan_USD
0,Regular,98312,22.74,45233,46.01,84910.35
1,Premium,205085,47.45,91808,44.77,85065.04
2,VIP,128859,29.81,50316,39.05,85025.33


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q3 - Nghịch lý Khách hàng VIP):**
> * **Nghịch lý được bóc tách:** Khách hàng **Regular** đạt CVR cao nhất (**46.01%**), Premium đạt **44.77%**, trong khi khách **VIP** lại có CVR thấp nhất sàn (**39.05%**), thấp hơn mức chuẩn sàn tới 4.29 điểm %.
> * **Giải thích chuyên sâu:** Khách VIP có thu nhập cao nhất nhưng không nhạy cảm với các đợt flash sale xả hàng đại trà. Quyết định: Cắt giảm ưu đãi giảm giá đại trà cho nhóm VIP, chuyển sang dịch vụ chăm sóc đặc quyền (White-glove Service).


---
## Truy vấn 4: Phân Hóa Hành Vi Mua Sắm Theo Nhóm Tuổi (Age Group Segmentation)

* **Câu hỏi phân tích (CMO Persona):** Độ tuổi khách hàng tác động như thế nào đến tỷ lệ chốt đơn trực tuyến?
* **Ý nghĩa đối với Bước 6:** Xác định biến `Age` mang hệ số tương quan âm ($r = -0.07$) trong ma trận đặc trưng.

In [5]:
q4 = """
SELECT 
    CASE 
        WHEN Age < 30 THEN '1. Young (<30 tuổi)'
        WHEN Age BETWEEN 30 AND 50 THEN '2. Middle (30-50 tuổi)'
        ELSE '3. Senior (>50 tuổi)'
    END AS Nhom_Tuoi,
    COUNT(*) AS So_Khach,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) AS Ty_Trong_Pct,
    SUM(PurchaseStatus) AS So_Khach_Mua,
    ROUND(AVG(PurchaseStatus) * 100.0, 2) AS Ty_Le_Chot_Don_CVR_Pct
FROM customers
GROUP BY Nhom_Tuoi
ORDER BY Nhom_Tuoi ASC;
"""
run_sql(q4)


,Nhom_Tuoi,So_Khach,Ty_Trong_Pct,So_Khach_Mua,Ty_Le_Chot_Don_CVR_Pct
0,1. Young (<30 tuổi),100033,23.14,47796,47.78
1,2. Middle (30-50 tuổi),169373,39.18,74723,44.12
2,3. Senior (>50 tuổi),162850,37.67,64838,39.81


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q4 - Nhóm tuổi):**
> * Khách hàng trẻ (<30 tuổi) đạt CVR cao nhất (**47.78%**), nhóm trung niên (30-50 tuổi) đạt **44.12%**, trong khi nhóm lớn tuổi (>50 tuổi) tụt dốc xuống **39.81%** (dưới mức chuẩn sàn).
> * **Chiến lược:** CMO cần điều chỉnh thông điệp tiếp thị nhắm mục tiêu: Chiến dịch trên TikTok/Reels dành cho giới trẻ, giao diện đơn giản phông chữ lớn dành cho khách hàng cao tuổi.


---
## Truy vấn 5: Đòn Bẩy Chương Trình Khách Hàng Thân Thiết (LoyaltyProgram Lift)

* **Câu hỏi nghiên cứu (RQ3 & Growth Lead Persona):** Thẻ hội viên thân thiết tạo ra mức tăng trưởng tỷ lệ chuyển đổi (Lift) là bao nhiêu phần trăm?
* **Ý nghĩa đối với Bước 6:** Biến có hệ số Odds Ratio cao nhất (**3.4793, $\\beta = 1.2468$** sau khi chuẩn hóa StandardScaler) trong mô hình hồi quy Logistic.

In [6]:
q5 = """
SELECT 
    CASE WHEN LoyaltyProgram = 1 THEN 'Có tham gia Loyalty' ELSE 'Không tham gia Loyalty' END AS Chuong_Trinh_Loyalty,
    COUNT(*) AS So_Khach,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) AS Ty_Trong_Pct,
    SUM(PurchaseStatus) AS So_Khach_Mua,
    ROUND(AVG(PurchaseStatus) * 100.0, 2) AS Ty_Le_Chot_Don_CVR_Pct
FROM customers
GROUP BY LoyaltyProgram
ORDER BY LoyaltyProgram DESC;
"""
run_sql(q5)


,Chuong_Trinh_Loyalty,So_Khach,Ty_Trong_Pct,So_Khach_Mua,Ty_Le_Chot_Don_CVR_Pct
0,Có tham gia Loyalty,219095,50.69,104072,47.50
1,Không tham gia Loyalty,213161,49.31,83285,39.07


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q5 - Hội viên Loyalty):**
> * Khách hàng tham gia Loyalty đạt CVR **47.50%**, cao hơn vượt bậc so với nhóm không tham gia là **39.07%** (tăng vọt **+8.43 điểm %**, tương đương +21.6% tương đối).
> * **Ứng dụng Kịch bản 2 (Bước 6):** Đối với khách hàng do dự ($30\% \le P < 70\%$), chính sách kích hoạt đặc quyền hội viên là đòn bẩy hiệu quả nhất để đẩy họ vượt ngưỡng chốt đơn.


---
## Truy vấn 6 (Window Function LAG): Tác Động & Ngưỡng Bão Hòa Của Mã Giảm Giá (DiscountsAvailed Elasticity)

* **Câu hỏi nghiên cứu (RQ3 & Growth Lead / CFO Persona):** Tặng bao nhiêu voucher thì đạt hiệu quả chuyển đổi tối ưu, và điểm bão hòa nằm ở đâu để tránh xói mòn lợi nhuận?
* **Kỹ thuật SQL nâng cao:** Ứng dụng hàm cửa sổ `LAG() OVER (ORDER BY ...)` để tính toán chính xác mức tăng chuyển đổi biên (Delta CVR) qua từng mã giảm giá.

In [7]:
q6 = """
WITH DiscountStats AS (
    SELECT 
        DiscountsAvailed AS So_Ma_Giam_Gia,
        COUNT(*) AS So_Khach,
        SUM(PurchaseStatus) AS So_Khach_Mua,
        ROUND(AVG(PurchaseStatus) * 100.0, 2) AS CVR_Hien_Tai
    FROM customers
    GROUP BY DiscountsAvailed
)
SELECT 
    So_Ma_Giam_Gia,
    So_Khach,
    ROUND(So_Khach * 100.0 / (SELECT COUNT(*) FROM customers), 2) AS Ty_Trong_Khach_Pct,
    So_Khach_Mua,
    CVR_Hien_Tai AS Ty_Le_Chot_Don_CVR_Pct,
    ROUND(CVR_Hien_Tai - LAG(CVR_Hien_Tai, 1) OVER (ORDER BY So_Ma_Giam_Gia), 2) AS Muc_Tang_Bien_Delta_CVR_Pct
FROM DiscountStats
ORDER BY So_Ma_Giam_Gia ASC;
"""
run_sql(q6)


,So_Ma_Giam_Gia,So_Khach,Ty_Trong_Khach_Pct,So_Khach_Mua,Ty_Le_Chot_Don_CVR_Pct,Muc_Tang_Bien_Delta_CVR_Pct
0,0,38432,8.89,15579,40.54,NaN
1,1,61635,14.26,24415,39.61,-0.93
2,2,66330,15.35,26210,39.51,-0.10
3,3,72631,16.80,31295,43.09,3.58
4,4,74008,17.12,34312,46.36,3.27
5,5,74518,17.24,34818,46.72,0.36
6,6,33421,7.73,15631,46.77,0.05
7,7,9123,2.11,4161,45.61,-1.16
8,8,1834,0.42,783,42.69,-2.92
9,9,272,0.06,122,44.85,2.16


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q6 - Điểm bão hòa Voucher):**
> * **Hiện tượng bão hòa kích cầu:** Tỷ lệ chốt đơn tăng liên tục từ 0 mã (**40.54%**) lên đỉnh tại 5 mã (**46.72%**). Từ mã thứ 6 trở đi, CVR đi ngang (**46.77%**) rồi giảm xuống **45.61%** (ở 7 mã) và **42.69%** (ở 8 mã).
> * **Thiết lập quy tắc cứng cho Bước 6 (Kịch bản 2):** Giới hạn tối đa không quá **5 voucher/khách hàng**. Việc cấp từ voucher thứ 6 trở đi không tạo thêm giá trị chuyển đổi mà chỉ gây lãng phí ngân sách tiếp thị.


---
## Truy vấn 7 (TRỌNG TÂM): Phân Tích Độ Trễ Giao Dịch (`LastPurchaseDaysAgo`) & Cơ Sở Bằng Chứng Cho Machine Learning

* **Vấn đề học thuật then chốt:** Chuẩn hóa ngưỡng độ trễ với mốc phân tách quyết định là **> 60 ngày** để xác định nhóm rời bỏ sàn.
* **Tác dụng đối với Tệp 6:** Cung cấp bằng chứng định lượng chính xác lý giải tại sao thuộc tính này có hệ số âm sâu nhất trong Logistic Regression ($\\beta = -8.0649$, Odds Ratio = 0.0003 sau khi chuẩn hóa StandardScaler), phân tách dứt khoát Kịch bản 1 ($P \ge 70\%$) và Kịch bản 3 ($P < 30\%$).

In [8]:
q7 = """
WITH RecencyBuckets AS (
    SELECT 
        CASE 
            WHEN LastPurchaseDaysAgo <= 15 THEN '1. <= 15 ngày (Mới mua gần đây)'
            WHEN LastPurchaseDaysAgo <= 30 THEN '2. 16 - 30 ngày (Trong 1 tháng)'
            WHEN LastPurchaseDaysAgo <= 60 THEN '3. 31 - 60 ngày (1 - 2 tháng)'
            ELSE '4. > 60 ngày (Nguy cơ rời bỏ / Churn)'
        END AS Nhom_Do_Tre_Recency,
        PurchaseStatus
    FROM customers
)
SELECT 
    Nhom_Do_Tre_Recency,
    COUNT(*) AS So_Luong_Khach,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) AS Ty_Trong_Tong_Khach_Pct,
    SUM(PurchaseStatus) AS So_Khach_Chot_Don,
    ROUND(AVG(PurchaseStatus) * 100.0, 2) AS Ty_Le_Chot_Don_CVR_Pct
FROM RecencyBuckets
GROUP BY Nhom_Do_Tre_Recency
ORDER BY Nhom_Do_Tre_Recency;
"""
run_sql(q7)


,Nhom_Do_Tre_Recency,So_Luong_Khach,Ty_Trong_Tong_Khach_Pct,So_Khach_Chot_Don,Ty_Le_Chot_Don_CVR_Pct
0,1. <= 15 ngày (Mới mua gần đây),105122,24.32,97840,93.07
1,2. 16 - 30 ngày (Trong 1 tháng),115058,26.62,78855,68.53
2,3. 31 - 60 ngày (1 - 2 tháng),51212,11.85,10554,20.61
3,4. > 60 ngày (Nguy cơ rời bỏ / Churn),160864,37.21,108,0.07


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q7 - Độ trễ giao dịch):**
> * **Hiện tượng sụt dốc đứng (Recency Cliff):**
>   * Nhóm mới mua $\le 15$ ngày: CVR đạt đỉnh **93.07%** (105.122 khách — nhóm trung tâm của Kịch bản 1).
>   * Nhóm 16–30 ngày: CVR đạt **68.53%** (115.058 khách).
>   * Nhóm 31–60 ngày: CVR rơi xuống **20.61%** (51.212 khách).
>   * Nhóm **> 60 ngày**: CVR rơi tự do xuống chỉ còn **0.07%** (160.864 khách chỉ có 108 khách mua — nhóm trung tâm của Kịch bản 3).
> * **Kết nối hoàn hảo sang Bước 6:** Khách hàng không mua sau 60 ngày gần như rời bỏ sàn hoàn toàn. Đây là căn cứ khoa học then chốt để Mô hình Học máy phân loại chính xác nhóm $P < 30\%$ nhằm ngắt ngay ngân sách quảng cáo trả phí (Paid Ads).


---
## Truy vấn 8 (Window Function NTILE 4): Phân Vị Thu Nhập và Sức Mua Thực Tế (AnnualIncome NTILE 4)

* **Câu hỏi nghiên cứu (RQ1 & CFO / CEO Persona):** Thu nhập có phải là động lực chuyển đổi đơn điệu làm tăng tỷ lệ mua hàng không?
* **Kỹ thuật SQL nâng cao:** Ứng dụng hàm cửa sổ `NTILE(4) OVER (ORDER BY AnnualIncome ASC)` chia đều 432.256 khách hàng thành 4 tứ phân vị thu nhập bằng nhau để triệt tiêu độ lệch mẫu.

In [9]:
q8 = """
WITH IncomeBuckets AS (
    SELECT 
        AnnualIncome,
        PurchaseStatus,
        NTILE(4) OVER (ORDER BY AnnualIncome ASC) AS Phan_Vi_Thu_Nhap
    FROM customers
)
SELECT 
    'Quartile ' || Phan_Vi_Thu_Nhap AS Nhom_Thu_Nhap_NTILE4,
    COUNT(*) AS So_Khach_Hang,
    ROUND(MIN(AnnualIncome), 0) AS Thu_Nhap_Thap_Nhat_USD,
    ROUND(MAX(AnnualIncome), 0) AS Thu_Nhap_Cao_Nhat_USD,
    ROUND(AVG(AnnualIncome), 0) AS Thu_Nhap_Binh_Quan_USD,
    SUM(PurchaseStatus) AS So_Khach_Chot_Don,
    ROUND(AVG(PurchaseStatus) * 100.0, 2) AS Ty_Le_Chot_Don_CVR_Pct
FROM IncomeBuckets
GROUP BY Phan_Vi_Thu_Nhap
ORDER BY Phan_Vi_Thu_Nhap ASC;
"""
run_sql(q8)


,Nhom_Thu_Nhap_NTILE4,So_Khach_Hang,Thu_Nhap_Thap_Nhat_USD,Thu_Nhap_Cao_Nhat_USD,Thu_Nhap_Binh_Quan_USD,So_Khach_Chot_Don,Ty_Le_Chot_Don_CVR_Pct
0,Quartile 1,108064,11966.0,51941.0,35131.0,40427,37.41
1,Quartile 2,108064,51941.0,83686.0,67511.0,44992,41.63
2,Quartile 3,108064,83686.0,116498.0,100072.0,48658,45.03
3,Quartile 4,108064,116499.0,204178.0,137358.0,53280,49.30


> **Kết luận nghiệp vụ & Tác dụng cho dự án (Q8 - Phân vị thu nhập NTILE 4):**
> * **Mối tương quan thuận đơn điệu hoàn hảo:**
>   * Quartile 1 ($11k–$51k): CVR đạt **37.41%**
>   * Quartile 2 ($51k–$83k): CVR đạt **41.63%**
>   * Quartile 3 ($83k–$116k): CVR đạt **45.03%**
>   * Quartile 4 ($116k–$204k): CVR đạt **49.30%** (chênh lệch **+11.89 điểm %** so với Q1).
> * **Bác bỏ hoàn toàn ảo giác EDA:** Khẳng định thu nhập tăng đều đặn làm tăng sức mua thực tế, chứng minh tính nhất quán xuyên suốt từ SQL sang Machine Learning.


In [10]:
# Đóng kết nối giải phóng tài nguyên nhưng bảo toàn CSDL đĩa bền vững Data/customers.db
conn.close()
print(" Đã hoàn tất thực thi hệ thống 8 truy vấn SQL tinh gọn chuẩn mực!")


 Đã hoàn tất thực thi hệ thống 8 truy vấn SQL tinh gọn chuẩn mực!


---
# PHẦN 3: BẢNG TỔNG HỢP 8 BẰNG CHỨNG SQL KẾT NỐI DÒNG CHẢY CRISP-DM

| Nhóm nội dung | Truy vấn SQL | Thuộc tính phân tích | Chỉ số định lượng nổi bật | Persona thụ hưởng | Kết nối sang Mô hình Học máy 3 Kịch bản (Tệp 6) |
|---|---|---|---|:---:|---|
| **PHẦN 1:<br>CÁC YẾU TỐ ĐI CÙNG VẤN ĐỀ GỐC** | **SQL 1** (1.1) | `PurchaseStatus` | **CVR chuẩn = 43.34%**, Thất thoát = 56.66% | Hội đồng & CEO | Mốc đối chuẩn cơ sở (Baseline Accuracy 56.66%). |
| | **SQL 2** (1.2) | `ReferralSource` | Paid Ads CVR thấp nhất (**43.05%**), Referral cao nhất (**44.00%**) | CMO | Ngắt ngân sách Paid Ads ở nhóm $P < 30\%$ (Kịch bản 3). |
| **PHẦN 2:<br>CÁC CÂU HỎI MỤC TIÊU KINH DOANH** | **SQL 3** (2.1) | `CustomerSegment` | Regular đạt CVR **46.01%**, VIP thấp nhất sàn (**39.05%**) | Growth Lead | Giải thích hệ số âm của biến VIP trong Logistic Regression: Odds Ratio = 0.4619 ($\\beta = -0.7725$). |
| | **SQL 4** (2.2) | `Age` | Giới trẻ (<30 tuổi) đạt CVR **47.78%**, lớn tuổi giảm còn **39.81%** | CMO | Feature nhân khẩu học đầu vào cho mô hình dự báo. |
| | **SQL 5** (2.3) | `LoyaltyProgram` | Có thẻ CVR **47.50%** vs không thẻ **39.07%** (+8.43 điểm %) | Growth Lead | **Odds Ratio = 3.4793 ($\\beta = 1.2468$)** (cao nhất). Đòn bẩy trọng tâm Kịch bản 2. |
| | **SQL 6** (2.4) | `DiscountsAvailed` | Đạt đỉnh CVR **46.72%** tại 5 mã, sau đó bão hòa/sụt giảm | CFO & Growth | Quy tắc chặn cứng tối đa **5 voucher/khách** cho Kịch bản 2. |
| | **SQL 7** (2.5) | LastPurchaseDaysAgo | $\le 15$ ngày CVR **93.07%**, $>60$ ngày CVR rơi về **0.07%** | Lead Data Scientist | Hệ số âm sâu nhất ($\\beta = -8.0649$, Odds Ratio = 0.0003). Tách nhóm Kịch bản 1 và 3. |
| | **SQL 8** (2.6) | `AnnualIncome` | Quartile 1 (**37.41%**) tăng đơn điệu lên Q4 (**49.30%**) | CFO & CEO | Động lực tài chính cốt lõi; biến liên tục phân loại khách tiềm năng. |

---
*Notebook tiếp theo: [4. Data Analysis with Python](./4.%20Data%20Analysis%20with%20Python.ipynb)*